In [1]:
from classifiers import *
from boilerplate import *

/home/smith.alyss/.conda/envs/deberta/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/smith.alyss/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
import pandas as pd
df_news = pd.read_csv('../Human/A1_r2.csv')
df_news['text_no_boilerplate'] = df_news['text'].apply(filter_boilerplate)
df_news['text_with_boilerplate'] = df_news['text']
df_news['text'] = df_news['text_no_boilerplate']

In [3]:
df_news['gold'].value_counts()
df_news['gold'] = df_news['gold'].fillna(0)

In [4]:
clf_mpox_in_kw = WordRelevanceClassifier()
clf_mpox_in_kw.accuracy_pr_kappa(df_news['keywords'], df_news['gold'])

{'accuracy': 0.944,
 'precision': 0.9405099150141643,
 'recall': 0.9793510324483776,
 'f1': 0.9595375722543352,
 'cohen_kappa': np.float64(0.868749179682373)}

In [5]:
clf_mpox_in_kw = WordRelevanceClassifier(threshold=0, source='keywords', keywords=["mpox", "monkeypox", "mpv", "mpx"])
clf_mpox_in_kw.accuracy_pr_kappa(df_news['keywords'], df_news['gold'])

{'accuracy': 0.95,
 'precision': 0.9385474860335196,
 'recall': 0.9911504424778761,
 'f1': 0.9641319942611191,
 'cohen_kappa': np.float64(0.8818257449705037)}

In [6]:
clf_kw = WordRelevanceClassifier(threshold=0, source='keywords', keywords=['monkeypox', 'transmission', 'transmit'])
clf_kw.accuracy_pr_kappa(df_news['keywords'], df_news['gold'])

{'accuracy': 0.942,
 'precision': 0.9378531073446328,
 'recall': 0.9793510324483776,
 'f1': 0.9581529581529582,
 'cohen_kappa': np.float64(0.8638344226579521)}

In [7]:
clf_briefing_in_title = WordRelevanceClassifier(threshold=0, source='text', keywords=['briefing'])
clf_briefing_in_title.accuracy_pr_kappa(df_news['title'], df_news['gold'])

/home/smith.alyss/.conda/envs/deberta/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


{'accuracy': 0.322,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'cohen_kappa': np.float64(0.0)}

In [8]:
clf_meta = MultiCriterionClassifier([(clf_briefing_in_title, False), (clf_kw, True)], threshold=1)
clf_meta.accuracy_pr_kappa((df_news['title'], df_news['keywords']), df_news['gold'])

{'accuracy': 0.942,
 'precision': 0.9378531073446328,
 'recall': 0.9793510324483776,
 'f1': 0.9581529581529582,
 'cohen_kappa': np.float64(0.8638344226579521)}

In [11]:
X_train_kw, X_test_kw, y_train_kw, y_test_kw = train_test_split(
    df_news['keywords'], 
    df_news['gold'], 
    test_size= 0.4,
)
clf_nb_kw = NBClassifier('keywords')
clf_nb_kw.fit(X_train_kw, y_train_kw)
clf_nb_kw.accuracy_pr_kappa(X_test_kw, y_test_kw)

{'accuracy': 0.865,
 'precision': 0.8407643312101911,
 'recall': 0.9850746268656716,
 'f1': 0.9072164948453608,
 'cohen_kappa': np.float64(0.6650955097990574)}

In [5]:
df_train, df_test = train_test_split(
    df_news, 
    test_size= 0.6,
)

In [13]:
clf_nb_txt = NBClassifier('text')
clf_nb_txt.fit(df_train['text'], df_train['gold'])
clf_nb_txt.accuracy_pr_kappa(df_test['text'], df_test['gold'])

{'accuracy': 0.9266666666666666,
 'precision': 0.9147982062780269,
 'recall': 0.9855072463768116,
 'f1': 0.9488372093023256,
 'cohen_kappa': np.float64(0.8200556191722559)}

In [14]:
clf_nb_title = NBClassifier('text')
clf_nb_title.fit(df_train['title'], df_train['gold'])
clf_nb_title.accuracy_pr_kappa(df_test['title'], df_test['gold'])

{'accuracy': 0.8,
 'precision': 0.779467680608365,
 'recall': 0.9903381642512077,
 'f1': 0.8723404255319149,
 'cohen_kappa': np.float64(0.4395665981692509)}

In [15]:
clf_meta = MultiCriterionClassifier([(clf_nb_txt, True), (clf_nb_title, True)], threshold=1)
clf_meta.accuracy_pr_kappa((df_test['text'], df_test['title']), df_test['gold'])

{'accuracy': 0.9333333333333333,
 'precision': 0.9308755760368663,
 'recall': 0.9758454106280193,
 'f1': 0.9528301886792453,
 'cohen_kappa': np.float64(0.839409025212783)}

In [16]:
test_labels = clf_meta.predict((df_test['text'], df_test['title']))
true_labels = df_test['gold']

In [16]:
from sklearn.metrics import confusion_matrix
confusion_matrix(true_labels, test_labels)

array([[ 72,  17],
       [  8, 203]])

In [ ]:
#Extrapolation to all data

In [ ]:
df_to_pred = pd.read_csv('..../Dataset/Pre-processing/mpox_news_data.csv')
df_to_pred['no_boilerplate'] = df_to_pred['text'].apply(filter_boilerplate)

In [30]:
df_to_pred['NB_relevance_label'] = clf_meta.predict((df_to_pred['no_boilerplate'], df_to_pred['title']))

In [ ]:
df_to_pred.to_csv('naive_bayes.csv')

In [ ]:
# df_train.to_csv('/work/netsi/smith.alyss/Framing_Mpox/Data/Processing/NB_train_val_test/train.csv')
# df_test.to_csv('/work/netsi/smith.alyss/Framing_Mpox/Data/Processing/NB_train_val_test/test.csv')


In [6]:
cols_for_x = ['title', 'text']
X_train_BERT, X_test_BERT, y_train_BERT, y_test_BERT = train_test_split(
    df_news[cols_for_x], 
    df_news['gold'], 
    test_size=0.4,
    random_state=5,
)
X_val_BERT, X_test_BERT, y_val_BERT, y_test_BERT = train_test_split(X_test_BERT, y_test_BERT, test_size=0.5, random_state=42, shuffle=True)

In [24]:
clf_bert = BERTClassifier('gold')
df_train = pd.concat((X_train_BERT, y_train_BERT), axis=1)
df_val = pd.concat((X_val_BERT, y_val_BERT), axis=1)
clf_bert.train_with_validation(df_train, df_val)
clf_bert.accuracy_pr_kappa(X_test_BERT, y_test_BERT)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 100/100 [00:00<00:00, 193.85 examples/s]
Detected kernel version 3.10.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Epoch,Training Loss,Validation Loss,F1
1,No log,0.632208,0.700000
2,No log,0.294985,0.962406
3,No log,0.362379,0.955224
4,0.389100,0.187006,0.977099
5,0.389100,0.315920,0.969697
6,0.389100,0.435416,0.955224
7,0.108200,0.260676,0.977099
8,0.108200,0.301328,0.969697
9,0.108200,0.303046,0.969697
10,0.015500,0.312590,0.969697


{'accuracy': 0.96,
 'precision': 0.9565217391304348,
 'recall': 0.9850746268656716,
 'f1': 0.9705882352941176,
 'cohen_kappa': np.float64(0.9081304547542489)}

In [8]:
# clf_deberta = DeBERTaClassifier('gold', 0)
# df_train = pd.concat((X_train_BERT, y_train_BERT), axis=1)
# df_val = pd.concat((X_val_BERT, y_val_BERT), axis=1)
# clf_deberta.train_with_validation(df_train, df_val)
clf_deberta.device = 'cuda:0'
clf_deberta.accuracy_pr_kappa(X_test_BERT, y_test_BERT)

{'accuracy': 0.98,
 'precision': 0.9850746268656716,
 'recall': 0.9850746268656716,
 'f1': 0.9850746268656716,
 'cohen_kappa': np.float64(0.9547715965626413)}

In [ ]:
df_to_pred = pd.read_csv('..../Dataset/Pre-processing/mpox_news_data.csv')
df_to_pred['text'] = df_to_pred['text'].apply(filter_boilerplate)
df_to_pred['DeBERTa_relevance_label'] = clf_deberta.predict((df_to_pred))

In [ ]:
df_to_pred.to_csv('deberta.csv')